# Quantum mechanics: wave packets, tunnelling and bound states

`ps.Schrodinger` solves the time-dependent Schrödinger equation

$$i\hbar\,\partial_t\psi = \Big[-\frac{\hbar^2}{2m}\nabla^2 + V(\mathbf x, t)\Big]\psi$$

for one particle on a 1D, 2D or 3D grid. On periodic grids it uses the **split-step Fourier** method (the kinetic
step is exact in Fourier space, so a free packet disperses exactly); on any grid it can use **Crank–Nicolson**
(hard walls with `boundary="dirichlet"`). Both are unitary: probability is conserved to round-off. Units here are
$\hbar = m = 1$ unless stated. The theory and the measured accuracy are in the
[theory note](../theory/quantum.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import physim as ps

## A free wave packet

A Gaussian packet $\psi \propto e^{-(x - x_0)^2/4\sigma_0^2}\,e^{ip_0x/\hbar}$ moves at $p_0/m$ and spreads:
$\sigma(t) = \sigma_0\sqrt{1 + (\hbar t / 2m\sigma_0^2)^2}$. A narrow packet has a wide spread of momenta, so it
disperses quickly. The grid is periodic and centred on zero; `run` records $\psi$ every `record_every` steps.

In [ ]:
s = ps.Schrodinger([2048], spacing=0.05)            # x in [-51.2, 51.2), periodic
s.set_gaussian(center=-20, width=1.0, momentum=2.0)
x = s.axes[0]
times, frames = s.run(dt=0.01, steps=1500, record_every=50)

density = np.abs(frames) ** 2
mean = density @ x / density.sum(axis=1)
width = np.sqrt(density @ x**2 / density.sum(axis=1) - mean**2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
for k, f in enumerate(range(0, len(times), 6)):
    ax1.plot(x, density[f], color=ps.plot.color(k), label=f"t = {times[f]:.0f}")
ax1.set_xlim(-28, 25); ax1.set_xlabel("x"); ax1.set_ylabel(r"$|\psi|^2$")
ps.plot.tidy(ax1)
ax2.plot(times, width, "o", ms=4, color=ps.plot.color(0), label="simulated")
ax2.plot(times, np.sqrt(1 + (times / 2) ** 2), color="black", lw=1, label="exact")
ax2.set_xlabel("t"); ax2.set_ylabel(r"$\sigma(t)$")
ps.plot.tidy(ax2)
plt.show()
print(f"width error {np.abs(width / np.sqrt(1 + (times / 2) ** 2) - 1).max():.1e}, "
      f"centre speed {np.polyfit(times, mean, 1)[0]:.10f}, norm {s.norm():.15f}")

The momentum distribution does not change for a free particle, only the phase of each component:

In [ ]:
p, phi0 = ps.quantum.momentum_density(s)
s.t = 0.0
s.set_gaussian(center=-20, width=1.0, momentum=2.0)
_, phi_start = ps.quantum.momentum_density(s)
print("largest change in |φ(p)|² since t = 0:", np.abs(phi0 - phi_start).max())

## Tunnelling through a barrier

A packet with energy below a rectangular barrier ($V_0 = 1$, width $a = 1$) is partly transmitted. The plots colour
$|\psi|^2$ by the phase of $\psi$ (hue): the transmitted packet keeps winding at the incoming wavelength, while on
the left the reflected wave interferes with the incoming one. The grey line is the barrier (right-hand axis).

Split-step needs a time step that resolves the shortest waves the grid holds when the potential has sharp edges:
$\Delta t \lesssim 2mh^2/\pi\hbar$ (here $h = 0.1$, so $\Delta t = 0.005$).

In [ ]:
V0, a = 1.0, 1.0
barrier = ps.Schrodinger([2048], spacing=0.1)       # x in [-102.4, 102.4)
xb = barrier.axes[0]
h = barrier.spacing
first = np.argmin(np.abs(xb + a / 2))               # exactly a / h cells wide
V = np.zeros_like(xb)
V[first:first + round(a / h)] = V0
barrier.potential = V
barrier.set_gaussian(center=-40, width=5, momentum=1.2)
print(f"mean energy {barrier.energy():.3f} (barrier height {V0})")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, t_show in zip(axes, [0, 30, 60]):
    barrier.step(0.005, int(round((t_show - barrier.t) / 0.005)))
    ps.plot.wavefunction(barrier, ax=ax)
    ax.set_xlim(-70, 50); ax.set_title(f"t = {barrier.t:.0f}")
fig.tight_layout()
plt.show()
transmitted = ps.quantum.probability(barrier, lambda x: x > a / 2)
print(f"transmitted {transmitted:.4f}, reflected {ps.quantum.probability(barrier, lambda x: x < -a / 2):.4f}")

The same run as an animation (`ps.plot.animate_wavefunction` works on what `run` records):

In [ ]:
barrier.t = 0.0
barrier.set_gaussian(center=-40, width=5, momentum=1.2)
t_anim, psi_anim = barrier.run(0.005, 14000, record_every=350)
anim = ps.plot.animate_wavefunction(t_anim, psi_anim, xb, potential=barrier.potential, interval=80)
HTML(anim.to_jshtml())

**Transmission against energy.** For a plane wave of energy $E$ the exact transmission probability is

$$T = \Big[1 + \frac{V_0^2\,\sinh^2(\kappa a)}{4E(V_0 - E)}\Big]^{-1}, \quad \kappa = \sqrt{2m(V_0 - E)}/\hbar
\qquad (E < V_0),$$

with $\sinh \to \sin$ above the barrier, where $T$ reaches 1 at resonances. A packet averages $T$ over its spread of
momenta (`ps.quantum.packet_transmission`); the simulated points match that average.

In [ ]:
momenta = np.linspace(0.7, 2.3, 9)
sigma = 5.0
simulated = []
for p0 in momenta:
    barrier.t = 0.0
    barrier.set_gaussian(center=-40, width=sigma, momentum=p0)
    barrier.step(0.005, int(round(70 / p0 / 0.005)))   # until the pieces have separated
    simulated.append(ps.quantum.probability(barrier, lambda x: x > a / 2))

E = np.linspace(0.05, 3.0, 400)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(E / V0, ps.quantum.barrier_transmission(E, V0, a), color="#999999", lw=1, label="plane wave")
p_fine = np.linspace(0.6, 2.4, 100)
ax.plot(p_fine**2 / 2 / V0, [ps.quantum.packet_transmission(p, sigma, V0, a) for p in p_fine],
        color="black", lw=1, label="packet average")
ax.plot(momenta**2 / 2 / V0, simulated, "o", color=ps.plot.color(1), label="simulated")
ax.axvline(1, color="#cccccc", lw=0.8)
ax.set_xlabel(r"$E / V_0$"); ax.set_ylabel("transmission")
ps.plot.tidy(ax)
plt.show()
expected = [ps.quantum.packet_transmission(p, sigma, V0, a) for p in momenta]
print("largest difference from the packet average:", np.abs(np.array(simulated) - expected).max())

## Bound states

`eigenstates(count)` finds the lowest eigenstates of the same discretised Hamiltonian the time stepping uses. For the
harmonic oscillator $V = \tfrac12 m\omega^2x^2$ the energies are $(n + \tfrac12)\hbar\omega$, and the spectral grid
gets them to round-off. Each state is drawn at its energy, inside the potential.

In [ ]:
osc = ps.Schrodinger([512], spacing=0.05, potential=lambda t, x: 0.5 * x**2)
energies, states = osc.eigenstates(6)
print("E_n - (n + 1/2):", energies - (np.arange(6) + 0.5))

xo = osc.axes[0]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(xo, osc.potential, color="#52514e", lw=1)
for n, (e, psi) in enumerate(zip(energies, states)):
    ax.axhline(e, color="#dddddd", lw=0.6)
    ax.plot(xo, e + 0.6 * psi, color=ps.plot.color(n), lw=1.2, label=f"n = {n}")
ax.set_xlim(-6, 6); ax.set_ylim(0, 7); ax.set_xlabel("x"); ax.set_ylabel("energy")
ps.plot.tidy(ax, legend=False)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()

**A double well and tunnelling oscillations.** In $V = \lambda(x^2 - x_0^2)^2$ the lowest states come in nearly
degenerate pairs: an even and an odd combination of states localised in each well, split by
$\Delta E = E_1 - E_0$. Starting in $(\psi_0 + \psi_1)/\sqrt2$, which sits in one well, the particle tunnels back
and forth with period $2\pi\hbar/\Delta E$: the probability in the right well is $\tfrac12(1 - \cos(\Delta E\,t/\hbar))$.

In [ ]:
lam, x0 = 0.3, 1.8
well = ps.Schrodinger([512], spacing=0.04, potential=lambda t, x: lam * (x**2 - x0**2) ** 2)
(e0, e1), (psi0, psi1) = well.eigenstates(2)
split = e1 - e0
print(f"barrier height {lam * x0**4:.3f}, E0 = {e0:.5f}, E1 = {e1:.5f}, tunnelling period {2 * np.pi / split:.1f}")

left = (psi0 - psi1) / np.sqrt(2)
if np.sum(left[well.axes[0] < 0] ** 2) < 0.5:   # pick the combination localised on the left
    left = (psi0 + psi1) / np.sqrt(2)
well.psi = left
t_w, psi_w = well.run(0.05, 4000, record_every=40)
xw = well.axes[0]
right = (np.abs(psi_w[:, xw > 0]) ** 2).sum(axis=1) * well.cell_volume

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.plot(xw, well.potential, color="#52514e", lw=1)
ax1.plot(xw, e0 + 2 * left**2, color=ps.plot.color(0), label="start")
ax1.set_xlim(-4, 4); ax1.set_ylim(0, 6); ax1.set_xlabel("x"); ax1.set_ylabel(r"$V$, $|\psi|^2$ (shifted)")
ps.plot.tidy(ax1)
ax2.plot(t_w, right, "o", ms=3, color=ps.plot.color(1), label="simulated")
ax2.plot(t_w, 0.5 * (1 - np.cos(split * t_w)), color="black", lw=1, label=r"$\frac{1}{2}(1 - \cos\Delta E t)$")
ax2.set_xlabel("t"); ax2.set_ylabel("probability in the right well")
ps.plot.tidy(ax2)
plt.show()

## The quantum-classical correspondence

Ehrenfest's theorem says $d\langle x\rangle/dt = \langle p\rangle/m$ and $d\langle p\rangle/dt = -\langle V'(x)\rangle$.
For a harmonic potential $\langle V'\rangle = V'(\langle x\rangle)$, so the centre of any packet moves exactly like a
classical particle. In an anharmonic potential that only holds while the packet is narrow compared with the scale on
which $V'$ changes. Here the double well $V = x^4/4 - x^2$ is run with a classical `World` and with packets for two
values of $\hbar$: the smaller $\hbar$ (a "more classical" particle, whose packet can be narrow in both position and
momentum) follows the classical orbit for longer.

In [ ]:
def v(t, x):
    return 0.25 * x**4 - x**2

w = ps.World(integrator="yoshida4")
w.add_particle([2.2, 0, 0])
w.add_force(ps.CustomForce(lambda t, pos, vel, m: np.c_[2 * pos[:, 0] - pos[:, 0] ** 3, np.zeros((len(pos), 2))],
                           velocity_dependent=False))
classical = w.run(0.002, 10000, record_every=50)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(classical.t, classical.pos[:, 0, 0], color="black", lw=1.5, label="classical")
for k, hbar in enumerate([0.2, 0.05]):
    q = ps.Schrodinger([1024], spacing=0.025, hbar=hbar, potential=v, order=4)
    q.set_gaussian(center=2.2, width=np.sqrt(hbar / 4), momentum=0.0)   # narrower for smaller ħ
    t_q, psi_q = q.run(0.002, 10000, record_every=50)
    d = np.abs(psi_q) ** 2
    ax.plot(t_q, d @ q.axes[0] / d.sum(axis=1), color=ps.plot.color(k), lw=1.2, label=rf"$\hbar = {hbar}$")
ax.set_xlabel("t"); ax.set_ylabel(r"$\langle x\rangle$")
ps.plot.tidy(ax)
plt.show()

## A double slit in two dimensions

A packet travelling in $+x$ meets a wall (a high potential) with two slits a distance $d = 5$ apart. What gets through
spreads from both slits and interferes: bright fringes sit where the path difference from the two slits is a whole
number of wavelengths $\lambda = 2\pi\hbar/p$. The edges of the grid carry an absorbing layer so nothing wraps around.
With one slit closed the fringes disappear.

In [ ]:
def slit_run(two_slits):
    s2 = ps.Schrodinger([256, 256], spacing=0.2)              # x, y in [-25.6, 25.6)
    X, Y = np.meshgrid(*s2.axes, indexing="ij")
    d, w_slit = 5.0, 1.0
    opening = np.abs(Y - d / 2) < w_slit / 2
    if two_slits:
        opening |= np.abs(Y + d / 2) < w_slit / 2
    s2.potential = np.where((np.abs(X) < 0.3) & ~opening, 30.0, 0.0)
    s2.absorbing_layer(5.0, 5.0)
    s2.set_gaussian(center=[-12, 0], width=[2.0, 3.0], momentum=[4.0, 0.0])
    s2.step(0.01, 700)
    return s2

two, one = slit_run(True), slit_run(False)
xs, ys = two.axes
screen = np.argmin(np.abs(xs - 12))
lam = 2 * np.pi / 4.0
# Bright fringes: |r1 - r2| = n λ from the slit centres (±d/2) to the screen at x = 12.
yy = np.linspace(-15, 15, 30001)
dr = np.hypot(xs[screen], yy - 2.5) - np.hypot(xs[screen], yy + 2.5)
bright = [yy[np.argmin(np.abs(dr - n * lam))] for n in range(-2, 3)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.2]})
ps.plot.wavefunction(two, ax=ax1, gamma=0.4)
ax1.axvline(xs[screen], color="white", lw=0.6, ls="--")
ax1.set_xlim(-20, 20); ax1.set_ylim(-15, 15); ax1.set_title("two slits, t = 7 (brightness ∝ |ψ|^0.8)")
ax2.plot(ys, two.density[screen], color=ps.plot.color(0), label="two slits")
ax2.plot(ys, one.density[screen], color=ps.plot.color(1), label="one slit")
for b in bright:
    ax2.axvline(b, color="#bbbbbb", lw=0.8)
ax2.set_xlim(-15, 15); ax2.set_xlabel("y on the screen (x = 12)"); ax2.set_ylabel(r"$|\psi|^2$")
ps.plot.tidy(ax2)
plt.show()
print(f"probability through the slits: two {ps.quantum.probability(two, lambda x, y: x > 0.5):.3f}, "
      f"one {ps.quantum.probability(one, lambda x, y: x > 0.5):.3f}")

## Where next

- Swap in your own potential: an array, or a function `V(t, x[, y[, z]])` for time-dependent problems (a driven
  trap, a moving barrier).
- `method="crank_nicolson"` and `boundary="dirichlet"` give a particle in a box with hard walls; its eigenstates
  converge to $n^2\pi^2\hbar^2/2mL^2$ as the grid is refined.
- `ps.quantum.probability_current(s)` gives $\mathbf j = (\hbar/m)\,\mathrm{Im}(\psi^*\nabla\psi)$, whose flux through
  a surface is the rate at which probability crosses it.